# 학생 실습용 코랩 | 단순 추세와 AI 단기 예측 비교

### 실행 준비
1. 노션에서 코랩을 열고 **파일 → Drive에 사본 저장**을 선택합니다.
2. **런타임 → 모두 실행**을 누르거나 각 셀 왼쪽 ▶를 위에서부터 누릅니다. CPU로 실행합니다.
3. CSV 자동 로딩이 되지 않으면 노션의 Drive CSV를 다운로드하고, 코랩 왼쪽 **파일 → 세션 저장소에 업로드**로 올린 뒤 다시 실행합니다. 파일 이름은 `daegu_143_daily_temperature.csv`로 유지합니다.

**완성 목표**는 2025년 예측값 CSV, MAE·RMSE 비교표, 그래프, 해석 3문장입니다.

## 실습 전에 확인
- 일평균기온(°C)은 같은 날짜의 시간 관측 기온을 평균한 값입니다.
- 관측 시간 수는 값의 품질을 확인하는 데 사용합니다. 24시각의 기온 중 하나라도 비어 있으면 불완전 관측일입니다.
- 이동평균은 최근 값들을 평균해 변화 흐름을 보는 지표입니다. 이번 실습의 선형 추세 예측은 최근 7일에 맞춘 직선을 하루 더 연장합니다.
- [강의용 코랩에서 지표와 그래프 해설 보기](https://colab.research.google.com/github/itlects/2026-2_ai-data/blob/main/week11-climate/01_example.ipynb) · [실습 CSV 직접 다운로드](https://drive.google.com/uc?export=download&id=1KDArjMKrEFZ4CY2nxvR-H8L6uUDV0Cma)


## 1. 데이터 읽고 품질 점검
1,826개 날짜를 읽습니다. 하루의 시간별 기온이 모두 있을 때만 실습에 사용합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

filename = "daegu_143_daily_temperature.csv"
paths = [Path(filename), Path("data") / filename]
url = "https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/week11-climate/data/" + filename
source = next((path for path in paths if path.is_file()), url)
raw = pd.read_csv(source)
display(raw.head())
print("관측 날짜 수", len(raw))


In [ ]:
raw["date"] = pd.to_datetime(raw["date"], errors="raise")
assert raw["date"].is_unique, "날짜가 중복되었습니다."
daily = raw.set_index("date").sort_index().asfreq("D")
incomplete = daily["hours_observed"].fillna(0).ne(24)
print("24시간 미만 관측일 수", incomplete.sum())
temp = daily["mean_temp_c"].where(~incomplete)
assert len(temp) == 1826


## 2. 입력과 정답 만들기
예측 대상 날짜를 D라고 하면 입력은 D−1~D−7 기온이며 정답은 D의 실제 기온입니다. 예측은 전날 관측 종료 후를 가정합니다. `shift(1)`은 바로 전날 값을 같은 행으로 옮깁니다. 결측으로 입력 또는 정답이 비어 있는 행은 제외합니다.

2021~2024년은 모델 학습, 2025년은 평가에 씁니다. 평가 중에도 예측일보다 앞선 실제 관측은 입력할 수 있습니다. 모델을 다시 학습하거나 예측값을 다음 입력으로 재사용하지 않습니다.

In [ ]:
frame = pd.DataFrame(index=temp.index)
lag_columns = [f"lag_{i}" for i in range(1, 8)]
for i, column in enumerate(lag_columns, start=1):
    frame[column] = temp.shift(i)

# 예측 대상 날짜 자체는 미리 알 수 있으므로 계절 특성으로 사용할 수 있습니다.
angle = 2 * np.pi * (frame.index.dayofyear - 1) / 365.25
frame["season_sin"] = np.sin(angle)
frame["season_cos"] = np.cos(angle)
frame["actual_c"] = temp
frame = frame.dropna()
train = frame.loc[:"2024-12-31"]
test = frame.loc["2025-01-01":]
assert train.index.max() < test.index.min()
print("학습", len(train), "평가", len(test))
display(test.head(3))


## 3. 두 방법으로 같은 날짜 예측
단순 추세는 직전 7일에 직선을 맞춰 다음 한 점까지 연장합니다. 랜덤 포레스트는 지난 기온과 계절 위치의 관계를 여러 결정 트리로 학습해 예측을 평균냅니다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 오래된 날부터 최근 날까지를 x=-7,...,-1에 놓고 x=0을 예측합니다.
days = np.arange(-7, 0)
history = test[lag_columns[::-1]].to_numpy()
slope, intercept = np.polyfit(days, history.T, 1)
features = lag_columns + ["season_sin", "season_cos"]
model = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=1)
model.fit(train[features], train["actual_c"])
results = pd.DataFrame({
    "actual_c": test["actual_c"],
    "trend_c": intercept,
    "forest_c": model.predict(test[features]),
}, index=test.index)
assert results.notna().all().all()


## 4. 오차 비교
MAE는 절대 오차의 평균이며, RMSE는 오차 제곱의 평균에 제곱근을 씌운 값입니다. 둘 다 °C 단위이며 낮을수록 오차가 작습니다. RMSE는 큰 오차에 더 민감합니다.

In [ ]:
model_columns = ["trend_c", "forest_c"]
metrics = pd.DataFrame({
    "MAE": [mean_absolute_error(results["actual_c"], results[c]) for c in model_columns],
    "RMSE": [np.sqrt(mean_squared_error(results["actual_c"], results[c])) for c in model_columns],
}, index=model_columns)
display(metrics.round(3))


## 5. 그래프로 차이를 확인하고 저장
그래프는 읽기 편하게 1~2월만 그립니다. 비교표와 예측 CSV는 평가에 사용한 2025년 전체 날짜를 대상으로 합니다. 코랩 왼쪽 파일 패널에서 생성된 세 파일을 다운로드합니다.

In [ ]:
ax = results.loc["2025-01-01":"2025-02-28"].plot(figsize=(11, 4))
ax.set(title="Daegu ASOS 143 | One-day-ahead forecasts", ylabel="Temperature (°C)", xlabel="Target date")
ax.figure.tight_layout()
ax.figure.savefig("week11_forecast_comparison.png", dpi=150)
plt.show()
results.to_csv("week11_predictions.csv", float_format="%.4f")
metrics.to_csv("week11_metrics.csv", float_format="%.4f")


## 6. 결과 작성
- 평가 행 수와 두 모델의 MAE·RMSE를 적습니다.
- MAE가 더 낮은 모델과 오차 차이를 적습니다.
- 두 모델 중 하나의 예측이 실제와 크게 달랐던 날짜를 그래프에서 찾고 CSV의 숫자를 근거로 설명합니다.
- 대구 한 관측소와 한 해의 평가 결과를 다른 지역이나 장기 기후 전망에 그대로 적용할 수 있는지 한 문장으로 답합니다.

제출물은 작성한 노트북, 예측 CSV, 평가 CSV, 그래프 PNG입니다.

### 실습 기록
- 24시간 미만 관측일 수 / 학습·평가 행 수
- 선형 추세 MAE·RMSE / 랜덤 포레스트 MAE·RMSE
- MAE가 낮은 방법과 두 MAE의 차이(°C)
- 오차가 컸던 날짜 / 실제 기온 / 선택한 모델의 예측 기온 / 절대 오차
- 이 평가 결과를 다른 지역 또는 장기 기후에 적용할 때의 한계
